In [3]:
import torch
print("CUDA Available:", torch.cuda.is_available())

CUDA Available: True


In [4]:
!pip install -q transformers accelerate bitsandbytes sentencepiece tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.9 MB/s eta 0:00:00


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os

BASE_DIR = "/content/drive/MyDrive/quantized"
os.makedirs(BASE_DIR, exist_ok=True)

In [9]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

fp16_path = os.path.join(BASE_DIR, "model-fp16")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model.save_pretrained(fp16_path)
tokenizer.save_pretrained(fp16_path)

print("FP16 saved.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FP16 saved.


In [12]:
from transformers import BitsAndBytesConfig

int8_path = os.path.join(BASE_DIR, "model-int8")

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.save_pretrained(int8_path)
tokenizer.save_pretrained(int8_path)

print("INT8 saved.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 saved.


In [13]:
int4_path = os.path.join(BASE_DIR, "model-int4")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

model.save_pretrained(int4_path)
tokenizer.save_pretrained(int4_path)

print("INT4 saved.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 saved.


In [14]:
import time

def get_size(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024**3)

def benchmark(path, name):
    print(f"\n===== {name} =====")

    model = AutoModelForCausalLM.from_pretrained(
        path,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(path)

    prompt = "Explain quantization in simple terms."
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    start = time.time()
    outputs = model.generate(**inputs, max_new_tokens=100)
    end = time.time()

    print("Time:", round(end-start,2),"sec")
    print("Size:", round(get_size(path),2),"GB")

benchmark(fp16_path, "FP16")
benchmark(int8_path, "INT8")
benchmark(int4_path, "INT4")


===== FP16 =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Time: 1.92 sec
Size: 2.05 GB

===== INT8 =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Time: 0.58 sec
Size: 1.15 GB

===== INT4 =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Time: 0.25 sec
Size: 0.71 GB


In [15]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!make

Cloning into 'llama.cpp'...
remote: Enumerating objects: 81343, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 81343 (delta 15), reused 4 (delta 4), pack-reused 81314 (from 2)
Receiving objects: 100% (81343/81343), 306.90 MiB | 14.08 MiB/s, done.
Resolving deltas: 100% (58859/58859), done.
/content/llama.cpp
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.


In [22]:
%cd /content/llama.cpp
!ls

/content/llama.cpp
AGENTS.md		       convert_lora_to_gguf.py	pocs
AUTHORS			       docs			poetry.lock
benches			       examples			pyproject.toml
build-xcframework.sh	       flake.lock		pyrightconfig.json
ci			       flake.nix		README.md
CLAUDE.md		       ggml			requirements
cmake			       gguf-py			requirements.txt
CMakeLists.txt		       grammars			scripts
CMakePresets.json	       include			SECURITY.md
CODEOWNERS		       LICENSE			src
common			       licenses			tests
CONTRIBUTING.md		       Makefile			tools
convert_hf_to_gguf.py	       media			vendor
convert_hf_to_gguf_update.py   models
convert_llama_ggml_to_gguf.py  mypy.ini


In [25]:
!python3 convert_hf_to_gguf.py \
  --outfile model-f16.gguf \
  --outtype f16 \
  /content/drive/MyDrive/quantized/model-fp16

INFO:hf-to-gguf:Loading model: model-fp16
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_output.weig

In [29]:
%cd /content/llama.cpp
!cmake -B build
!cmake --build build --config Release

/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend


In [30]:
!ls build/bin

libggml-base.so		       llama-quantize
libggml-base.so.0	       llama-qwen2vl-cli
libggml-base.so.0.9.7	       llama-retrieval
libggml-cpu.so		       llama-save-load-state
libggml-cpu.so.0	       llama-server
libggml-cpu.so.0.9.7	       llama-simple
libggml.so		       llama-simple-chat
libggml.so.0		       llama-speculative
libggml.so.0.9.7	       llama-speculative-simple
libllama.so		       llama-tokenize
libllama.so.0		       llama-tts
libllama.so.0.0.8185	       llama-vdot
libmtmd.so		       test-alloc
libmtmd.so.0		       test-arg-parser
libmtmd.so.0.0.8185	       test-autorelease
llama-batched		       test-backend-ops
llama-batched-bench	       test-backend-sampler
llama-bench		       test-barrier
llama-cli		       test-c
llama-completion	       test-chat
llama-convert-llama2c-to-ggml  test-chat-parser
llama-cvector-generator        test-chat-peg-parser
llama-debug		       test-chat-template
llama-diffusion-cli	       test-gbnf-validator
llama-embedding		       test-gguf
llama-eva

In [31]:
!./build/bin/llama-quantize \
  model-f16.gguf \
  model-q4_0.gguf \
  q4_0

main: build = 8185 (2afcdb977)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-f16.gguf' to 'model-q4_0.gguf' as Q4_0
llama_model_loader: loaded meta data with 30 key-value pairs and 201 tensors from model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Model Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6:                     l

In [32]:
!mv model-q4_0.gguf /content/drive/MyDrive/quantized/model.gguf

In [40]:
import os
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_DIR = "/content/drive/MyDrive/quantized"

fp16_path = os.path.join(BASE_DIR, "model-fp16")
int8_path = os.path.join(BASE_DIR, "model-int8")
int4_path = os.path.join(BASE_DIR, "model-int4")
gguf_path = os.path.join(BASE_DIR, "model.gguf")

print("Paths verified.")

Paths verified.


In [41]:
def get_size(path):
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / (1024**3)

print("FP16 Size:", round(get_size(fp16_path), 2), "GB")
print("INT8 Size:", round(get_size(int8_path), 2), "GB")
print("INT4 Size:", round(get_size(int4_path), 2), "GB")

FP16 Size: 2.05 GB
INT8 Size: 1.15 GB
INT4 Size: 0.71 GB


In [42]:
gguf_size = os.path.getsize(gguf_path) / (1024**3)
print("GGUF Size:", round(gguf_size, 2), "GB")

GGUF Size: 0.59 GB


In [51]:
PROMPT = "What does RGB means"

def benchmark(model_path, name):
    print(f"\n===== {name} =====")

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    inputs = tokenizer(PROMPT, return_tensors="pt").to("cuda")

    torch.cuda.synchronize()
    start = time.time()

    outputs = model.generate(**inputs, max_new_tokens=100)

    torch.cuda.synchronize()
    end = time.time()

    print("Time:", round(end - start, 2), "seconds")

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("Output Preview:\n", result[:300])

benchmark(fp16_path, "FP16")
benchmark(int8_path, "INT8")
benchmark(int4_path, "INT4")


===== FP16 =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Time: 0.23 seconds
Output Preview:
 What does RGB means in the context of color theory?

===== INT8 =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Time: 1.16 seconds
Output Preview:
 What does RGB means in the context of color theory?

===== INT4 =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Time: 0.52 seconds
Output Preview:
 What does RGB means in the context of color theory?


In [52]:
%cd /content/llama.cpp

/content/llama.cpp


In [55]:
!./build/bin/llama-cli \
  -m /content/drive/MyDrive/quantized/model.gguf \
  -p "What does RGB means" \
  -n 100


Loading model... |-\|/-\|/-\|/-\|/ 


▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b8185-2afcdb977
model      : model.gguf
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read               add a text file


> What does RGB means

|-\|/-\|/ RGB is an acronym that stands for Red, Green, and Blue, and it refers to the primary colors that are commonly used in color-coding, electronics, and photography. In simple terms, RGB represents the three primary colors: red, green, and blue. These colors are used to create all other colors in the visible spectrum, which is why they are commonly used in color schemes and color printing.

